In [2]:
"""
Experiment 1 — Does behavior survive quantization?
Models:  GPT-2 Small at FP32, INT8, INT4 (fake-quantized weights)
Tasks:   (a) Induction copy accuracy  (b) IOI logit difference
Run:     python exp1_behavior.py   (~15-20 min on a 3090)
"""

import torch
import numpy as np
import pandas as pd
from pathlib import Path
from transformer_lens import HookedTransformer

SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)

# ---------- 1. Fake quantization ----------

def quantize_tensor(w: torch.Tensor, n_bits: int) -> torch.Tensor:
    """Symmetric per-channel round-to-nearest (RTN)."""
    if w.ndim == 1:
        scale = w.abs().max() / (2 ** (n_bits - 1) - 1)
        scale = max(scale, 1e-12)
        return (w / scale).round().clamp(-(2 ** (n_bits - 1)), 2 ** (n_bits - 1) - 1) * scale
    scale = w.abs().amax(dim=tuple(range(w.ndim - 1)), keepdim=True) / (2 ** (n_bits - 1) - 1)
    scale = scale.clamp(min=1e-12)
    q = (w / scale).round().clamp(-(2 ** (n_bits - 1)), 2 ** (n_bits - 1) - 1)
    return q * scale


QUANT_TARGETS = ("W_Q", "W_K", "W_V", "W_O", "W_in", "W_out")
# embeddings, unembedding, layernorm, biases stay full precision
# (matches what bitsandbytes/GPTQ/AWQ quantize in practice)


def make_model(n_bits=None) -> HookedTransformer:
    model = HookedTransformer.from_pretrained("gpt2").to(DEVICE)
    if n_bits is None:
        return model
    with torch.no_grad():
        n_quantized = 0
        for name, param in model.named_parameters():
            if any(t in name for t in QUANT_TARGETS):
                param.copy_(quantize_tensor(param.data, n_bits))
                n_quantized += 1
    print(f"  quantized {n_quantized} weight tensors to INT{n_bits}")
    return model


# ---------- 2. Task A: Induction ----------

def induction_score(model, n_prompts=200, seq_len=50):
    """Random sequence repeated twice; second half should be predictable
    by copying. Report top-1 copy accuracy + loss on the repeated half."""
    g = torch.Generator(device="cpu").manual_seed(SEED)
    vocab = model.cfg.d_vocab
    correct, total, losses = 0, 0, []
    for i in range(0, n_prompts, 50):
        batch = min(50, n_prompts - i)
        seq = torch.randint(1000, vocab - 1000, (batch, seq_len), generator=g)
        tokens = torch.cat([seq, seq], dim=1).to(DEVICE)
        with torch.no_grad():
            logits = model(tokens)
        preds = logits[:, seq_len:-1].argmax(-1)
        targets = tokens[:, seq_len + 1:]
        correct += (preds == targets).sum().item()
        total += targets.numel()
        logp = torch.log_softmax(logits[:, seq_len:-1], dim=-1)
        losses.append(-logp.gather(-1, targets.unsqueeze(-1)).mean().item())
    return {"induction_acc": correct / total, "induction_loss": float(np.mean(losses))}


# ---------- 3. Task B: IOI ----------

NAMES = ["Mary", "John", "Tom", "James", "Dan", "Sarah", "Alice", "Ruth",
         "Peter", "Anna", "Paul", "Laura", "Mark", "Emma", "Kevin", "Lisa"]
TEMPLATES = [
    "When {A} and {B} went to the store, {B} gave a drink to",
    "When {A} and {B} went to the park, {B} handed a ball to",
    "Then {A} and {B} were at the school, and {B} gave a book to",
    "After {A} and {B} left the bar, {B} passed the keys to",
]


def ioi_prompts(n=500):
    rng = np.random.default_rng(SEED)
    out = []
    while len(out) < n:
        a, b = rng.choice(NAMES, size=2, replace=False)
        t = TEMPLATES[rng.integers(len(TEMPLATES))]
        out.append((t.format(A=a, B=b), " " + a, " " + b))
    return out


def ioi_score(model, n=500):
    """Mean logit(IO) - logit(S) at final position. Positive = correct name preferred."""
    prompts = ioi_prompts(n)
    diffs = []
    for i in range(0, n, 50):
        chunk = prompts[i:i + 50]
        for text, io, s in chunk:
            tokens = model.to_tokens(text)
            with torch.no_grad():
                logits = model(tokens)
            row = logits[0, -1]
            io_tok = model.to_single_token(io)
            s_tok = model.to_single_token(s)
            diffs.append((row[io_tok] - row[s_tok]).item())
    diffs = np.array(diffs)
    return {"ioi_logit_diff": float(diffs.mean()),
            "ioi_logit_diff_std": float(diffs.std()),
            "ioi_acc": float((diffs > 0).mean())}


# ---------- 4. Run ----------

def main():
    rows = []
    for label, bits in [("FP32", None), ("INT8", 8), ("INT4", 4)]:
        print(f"\n=== {label} ===")
        model = make_model(bits)
        row = {"precision": label}
        row.update(induction_score(model))
        row.update(ioi_score(model))
        rows.append(row)
        print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})
        del model
        torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_DIR / "exp1_results.csv", index=False)
    print("\n===== EXPERIMENT 1 — BEHAVIORAL SURVIVAL =====")
    print(df.to_string(index=False))


if __name__ == "__main__":
    main()


=== FP32 ===


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
{'precision': 'FP32', 'induction_acc': 0.9864, 'induction_loss': 0.247, 'ioi_logit_diff': 3.1845, 'ioi_logit_diff_std': 1.4421, 'ioi_acc': 0.992}

=== INT8 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
  quantized 72 weight tensors to INT8
{'precision': 'INT8', 'induction_acc': 0.9866, 'induction_loss': 0.2425, 'ioi_logit_diff': 3.3153, 'ioi_logit_diff_std': 1.4876, 'ioi_acc': 0.992}

=== INT4 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
  quantized 72 weight tensors to INT4
{'precision': 'INT4', 'induction_acc': 0.0034, 'induction_loss': 12.6783, 'ioi_logit_diff': -0.3132, 'ioi_logit_diff_std': 1.4128, 'ioi_acc': 0.442}

===== EXPERIMENT 1 — BEHAVIORAL SURVIVAL =====
precision  induction_acc  induction_loss  ioi_logit_diff  ioi_logit_diff_std  ioi_acc
     FP32       0.986429        0.246978        3.184467            1.4421

In [3]:
"""
Exp 1b — Group-wise RTN quantization, bit sweep 8..4
Group-wise = one scale per 128 input channels per output channel.
This is what real INT4 deployments do. Much gentler than per-channel.
"""
import torch, numpy as np, pandas as pd
from transformer_lens import HookedTransformer

SEED = 0
DEVICE = "cuda"
torch.manual_seed(SEED); np.random.seed(SEED)

GROUP = 128
QUANT_TARGETS = ("W_Q", "W_K", "W_V", "W_O", "W_in", "W_out")

def quantize_groupwise(w, n_bits, group=GROUP):
    """Group-wise symmetric RTN along the input dimension (dim=-2)."""
    orig_shape = w.shape
    d_in = w.shape[-2]
    if d_in % group != 0:          # fallback: per-channel
        scale = w.abs().amax(dim=-2, keepdim=True) / (2**(n_bits-1)-1)
        scale = scale.clamp(min=1e-12)
        return ((w/scale).round().clamp(-(2**(n_bits-1)), 2**(n_bits-1)-1))*scale
    n_groups = d_in // group
    w2 = w.reshape(*w.shape[:-2], n_groups, group, w.shape[-1])
    scale = w2.abs().amax(dim=-2, keepdim=True) / (2**(n_bits-1)-1)
    scale = scale.clamp(min=1e-12)
    q = (w2/scale).round().clamp(-(2**(n_bits-1)), 2**(n_bits-1)-1) * scale
    return q.reshape(orig_shape)

def make_model(n_bits=None):
    model = HookedTransformer.from_pretrained("gpt2").to(DEVICE)
    if n_bits is None: return model
    with torch.no_grad():
        for name, p in model.named_parameters():
            if any(t in name for t in QUANT_TARGETS):
                p.copy_(quantize_groupwise(p.data, n_bits))
    return model

# --- reuse induction_score and ioi_score from exp1 (paste them here) ---

rows = []
for label, bits in [("FP32", None), ("INT8", 8), ("INT7", 7), ("INT6", 6),
                    ("INT5", 5), ("INT4", 4)]:
    print(f"=== {label} ===")
    model = make_model(bits)
    row = {"precision": label}
    row.update(induction_score(model))
    row.update(ioi_score(model))
    rows.append(row); print(row)
    del model; torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv("results/exp1b_sweep.csv", index=False)
print(df.to_string(index=False))

=== FP32 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
{'precision': 'FP32', 'induction_acc': 0.9864285714285714, 'induction_loss': 0.24697836861014366, 'ioi_logit_diff': 3.184467269897461, 'ioi_logit_diff_std': 1.4421184805980833, 'ioi_acc': 0.992}
=== INT8 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
{'precision': 'INT8', 'induction_acc': 0.9865306122448979, 'induction_loss': 0.23942595720291138, 'ioi_logit_diff': 3.3039973888397216, 'ioi_logit_diff_std': 1.458284113528556, 'ioi_acc': 0.994}
=== INT7 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
{'precision': 'INT7', 'induction_acc': 0.9861224489795918, 'induction_loss': 0.2742903381586075, 'ioi_logit_diff': 3.068813049316406, 'ioi_logit_diff_std': 1.3471831503891993, 'ioi_acc': 0.994}
=== INT6 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
{'precision': 'INT6', 'induction_acc

In [4]:
"""
Exp 2 — Circuit identity under quantization (IOI, activation patching)
For each precision: patch each head's output (clean -> corrupted run),
measure how much of the correct behavior it restores.
Output: 12x12 head-importance map per precision + similarity to FP32 map.
"""
import torch, numpy as np, pandas as pd
from functools import partial
from transformer_lens import HookedTransformer

SEED = 0
DEVICE = "cuda"
torch.manual_seed(SEED); np.random.seed(SEED)

N_PAIRS = 25   # prompt pairs (raise to 50 for the paper version)
NAMES = ["Mary","John","Tom","James","Dan","Sarah","Alice","Ruth",
         "Peter","Anna","Paul","Laura","Mark","Emma","Kevin","Lisa"]
TEMPLATES = [
    "When {A} and {B} went to the store, {B} gave a drink to",
    "When {A} and {B} went to the park, {B} handed a ball to",
]

def make_pairs(n=N_PAIRS):
    """clean: answer=A. corrupted: names swapped -> answer=B."""
    rng = np.random.default_rng(SEED); out = []
    while len(out) < n:
        a, b = rng.choice(NAMES, size=2, replace=False)
        t = TEMPLATES[rng.integers(len(TEMPLATES))]
        out.append((t.format(A=a,B=b), t.format(A=b,B=a), " "+a, " "+b))
    return out

def logit_diff(logits, io_toks, s_toks):
    """mean over batch of logit(IO) - logit(S) at final position"""
    final = logits[:, -1]
    idx = torch.arange(final.shape[0])
    return (final[idx, io_toks] - final[idx, s_toks]).mean()

def patching_map(model):
    pairs = make_pairs()
    clean_toks  = model.to_tokens([p[0] for p in pairs])
    corr_toks   = model.to_tokens([p[1] for p in pairs])
    io = torch.tensor([model.to_single_token(p[2]) for p in pairs], device=DEVICE)
    s  = torch.tensor([model.to_single_token(p[3]) for p in pairs], device=DEVICE)

    with torch.no_grad():
        clean_logits, clean_cache = model.run_with_cache(clean_toks)
        corr_logits = model(corr_toks)
    clean_ld = logit_diff(clean_logits, io, s).item()
    corr_ld  = logit_diff(corr_logits,  io, s).item()
    print(f"  clean logit diff {clean_ld:.3f} | corrupted {corr_ld:.3f}")

    def patch_head(z, hook, head, layer):
        z[:, :, head, :] = clean_cache[f"blocks.{layer}.attn.hook_z"][:, :, head, :]
        return z

    n_l, n_h = model.cfg.n_layers, model.cfg.n_heads
    result = np.zeros((n_l, n_h))
    for layer in range(n_l):
        for head in range(n_h):
            hook_fn = partial(patch_head, head=head, layer=layer)
            with torch.no_grad():
                patched = model.run_with_hooks(
                    corr_toks,
                    fwd_hooks=[(f"blocks.{layer}.attn.hook_z", hook_fn)])
            ld = logit_diff(patched, io, s).item()
            # 0 = no effect, 1 = fully restores clean behavior
            result[layer, head] = (ld - corr_ld) / (clean_ld - corr_ld + 1e-9)
        print(f"  layer {layer} done")
    return result, clean_ld, corr_ld

def top_heads(m, k=10):
    flat = [(-abs(m[l,h]), l, h, m[l,h]) for l in range(12) for h in range(12)]
    return [(l,h,round(v,3)) for _,l,h,v in sorted(flat)[:k]]

maps = {}
for label, bits in [("FP32", None), ("INT8", 8), ("INT6", 6), ("INT5", 5), ("INT4", 4)]:
    print(f"=== {label} ===")
    model = make_model(bits)          # from exp1b (group-wise)
    m, cl, co = patching_map(model)
    maps[label] = m
    np.save(f"results/exp2_map_{label}.npy", m)
    print(f"  top heads: {top_heads(m)}")
    del model; torch.cuda.empty_cache()

# --- compare maps to FP32 ---
print("\n===== CIRCUIT IDENTITY vs FP32 =====")
fp = maps["FP32"].flatten()
for label in maps:
    if label == "FP32": continue
    q = maps[label].flatten()
    corr = np.corrcoef(fp, q)[0,1]
    top_fp = set((l,h) for l,h,_ in top_heads(maps["FP32"]))
    top_q  = set((l,h) for l,h,_ in top_heads(maps[label]))
    jac = len(top_fp & top_q) / len(top_fp | top_q)
    print(f"{label}: map correlation {corr:.3f} | top-10 head overlap {jac:.2f}")

=== FP32 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
  clean logit diff 4.024 | corrupted -3.966
  layer 0 done
  layer 1 done
  layer 2 done
  layer 3 done
  layer 4 done
  layer 5 done
  layer 6 done
  layer 7 done
  layer 8 done
  layer 9 done
  layer 10 done
  layer 11 done
  top heads: [(10, 7, np.float64(-0.532)), (11, 10, np.float64(-0.271)), (9, 9, np.float64(0.227)), (8, 10, np.float64(0.158)), (10, 0, np.float64(0.137)), (7, 9, np.float64(0.097)), (10, 6, np.float64(0.085)), (4, 11, np.float64(0.077)), (11, 2, np.float64(-0.066)), (8, 3, np.float64(0.06))]
=== INT8 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
  clean logit diff 4.126 | corrupted -4.074
  layer 0 done
  layer 1 done
  layer 2 done
  layer 3 done
  layer 4 done
  layer 5 done
  layer 6 done
  layer 7 done
  layer 8 done
  layer 9 done
  layer 10 done
  layer 11 done
  top heads: [(10, 7, np.float64(-0.517)), (11, 10, np.float64(-

In [5]:
"""
Exp 2b — Robustness: does the INT4 backup-head effect survive
100 pairs, 3 seeds, 4 templates?
Also fixes the normalization concern: reports RAW logit-diff restoration
alongside normalized, so INT4's smaller spread can't fake the effect.
"""
import torch, numpy as np, pandas as pd
from functools import partial
from transformer_lens import HookedTransformer

DEVICE = "cuda"
N_PAIRS = 100
SEEDS = [0, 1, 2]
NAMES = ["Mary","John","Tom","James","Dan","Sarah","Alice","Ruth",
         "Peter","Anna","Paul","Laura","Mark","Emma","Kevin","Lisa"]
TEMPLATES = [
    "When {A} and {B} went to the store, {B} gave a drink to",
    "When {A} and {B} went to the park, {B} handed a ball to",
    "Then {A} and {B} were at the school, and {B} gave a book to",
    "After {A} and {B} left the bar, {B} passed the keys to",
]

# heads we're tracking (from Wang et al. + our exp2)
TRACKED = {"main_movers": [(9,9),(9,6),(10,0)],
           "backup_movers": [(9,7),(10,2),(10,10),(11,2),(10,6),(10,1),(11,9)],
           "negative": [(10,7),(11,10)]}

def make_pairs(n, seed):
    rng = np.random.default_rng(seed); out = []
    while len(out) < n:
        a, b = rng.choice(NAMES, size=2, replace=False)
        t = TEMPLATES[rng.integers(len(TEMPLATES))]
        out.append((t.format(A=a,B=b), t.format(A=b,B=a), " "+a, " "+b))
    return out

def logit_diff(logits, io_toks, s_toks):
    final = logits[:, -1]
    idx = torch.arange(final.shape[0])
    return (final[idx, io_toks] - final[idx, s_toks]).mean()

def patching_map(model, seed):
    pairs = make_pairs(N_PAIRS, seed)
    # batch in chunks of 25 to be safe on memory
    n_l, n_h = model.cfg.n_layers, model.cfg.n_heads
    raw = np.zeros((n_l, n_h)); norm = np.zeros((n_l, n_h))
    clean_lds, corr_lds = [], []
    all_chunks = [pairs[i:i+25] for i in range(0, N_PAIRS, 25)]
    chunk_data = []
    for chunk in all_chunks:
        ct  = model.to_tokens([p[0] for p in chunk])
        cot = model.to_tokens([p[1] for p in chunk])
        io = torch.tensor([model.to_single_token(p[2]) for p in chunk], device=DEVICE)
        s  = torch.tensor([model.to_single_token(p[3]) for p in chunk], device=DEVICE)
        with torch.no_grad():
            cl_logits, cl_cache = model.run_with_cache(ct)
            co_logits = model(cot)
        chunk_data.append((ct, cot, io, s, cl_cache))
        clean_lds.append(logit_diff(cl_logits, io, s).item())
        corr_lds.append(logit_diff(co_logits, io, s).item())
    clean_ld, corr_ld = np.mean(clean_lds), np.mean(corr_lds)

    def patch_head(z, hook, head, layer, cache):
        z[:, :, head, :] = cache[f"blocks.{layer}.attn.hook_z"][:, :, head, :]
        return z

    for layer in range(n_l):
        for head in range(n_h):
            lds = []
            for (ct, cot, io, s, cl_cache) in chunk_data:
                hook_fn = partial(patch_head, head=head, layer=layer, cache=cl_cache)
                with torch.no_grad():
                    patched = model.run_with_hooks(
                        cot, fwd_hooks=[(f"blocks.{layer}.attn.hook_z", hook_fn)])
                lds.append(logit_diff(patched, io, s).item())
            ld = np.mean(lds)
            raw[layer, head]  = ld - corr_ld                       # raw restoration
            norm[layer, head] = (ld - corr_ld) / (clean_ld - corr_ld + 1e-9)
        print(f"    layer {layer} done")
    return raw, norm, clean_ld, corr_ld

def report_tracked(m):
    out = {}
    for group, heads in TRACKED.items():
        out[group] = {f"{l}.{h}": round(float(m[l,h]),3) for l,h in heads}
    return out

results = {}
for label, bits in [("FP32", None), ("INT4", 4)]:
    for seed in SEEDS:
        print(f"=== {label} seed {seed} ===")
        model = make_model(bits)   # group-wise version from exp1b
        raw, norm, cl, co = patching_map(model, seed)
        results[(label, seed)] = (raw, norm)
        np.save(f"results/exp2b_raw_{label}_s{seed}.npy", raw)
        np.save(f"results/exp2b_norm_{label}_s{seed}.npy", norm)
        del model; torch.cuda.empty_cache()

# ---- summary: mean over seeds for tracked heads ----
print("\n===== TRACKED HEADS (RAW restoration, mean±std over seeds) =====")
for label in ["FP32", "INT4"]:
    print(f"\n--- {label} ---")
    for group, heads in TRACKED.items():
        for (l, h) in heads:
            vals = [results[(label,s)][0][l,h] for s in SEEDS]
            print(f"  {group:15s} {l}.{h}: {np.mean(vals):+.3f} ± {np.std(vals):.3f}")

print("\n===== MAP SIMILARITY INT4 vs FP32 (normalized, per seed) =====")
for s in SEEDS:
    fp = results[("FP32",s)][1].flatten(); q = results[("INT4",s)][1].flatten()
    print(f"seed {s}: correlation {np.corrcoef(fp,q)[0,1]:.3f}")

=== FP32 seed 0 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
    layer 0 done
    layer 1 done
    layer 2 done
    layer 3 done
    layer 4 done
    layer 5 done
    layer 6 done
    layer 7 done
    layer 8 done
    layer 9 done
    layer 10 done
    layer 11 done
=== FP32 seed 1 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
    layer 0 done
    layer 1 done
    layer 2 done
    layer 3 done
    layer 4 done
    layer 5 done
    layer 6 done
    layer 7 done
    layer 8 done
    layer 9 done
    layer 10 done
    layer 11 done
=== FP32 seed 2 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
    layer 0 done
    layer 1 done
    layer 2 done
    layer 3 done
    layer 4 done
    layer 5 done
    layer 6 done
    layer 7 done
    layer 8 done
    layer 9 done
    layer 10 done
    layer 11 done
=== INT4 seed 0 ===
Loaded pretrained model gpt2 into HookedTransformer
Movi

In [6]:
"""
Exp 3 (FIXED v2) — self-contained. Correct final-position readout via
explicit prompt lengths. FP32 sanity gate runs first, automatically.
Needs only: make_model + quantize_groupwise (from exp1b) in scope.
"""
import torch, numpy as np
from functools import partial
from scipy import stats

DEVICE = "cuda"
N_PAIRS = 100
SEEDS = [0, 1, 2]
NAMES = ["Mary","John","Tom","James","Dan","Sarah","Alice","Ruth",
         "Peter","Anna","Paul","Laura","Mark","Emma","Kevin","Lisa"]
TEMPLATES = [
    "When {A} and {B} went to the store, {B} gave a drink to",
    "When {A} and {B} went to the park, {B} handed a ball to",
    "Then {A} and {B} were at the school, and {B} gave a book to",
    "After {A} and {B} left the bar, {B} passed the keys to",
]
POSITIVE = [(9,9),(10,0),(9,7),(10,10),(10,6)]
NEGATIVE = [(10,7),(11,10),(11,2),(10,2)]
ALL_TRACKED = POSITIVE + NEGATIVE

def make_pairs(n, seed):
    rng = np.random.default_rng(seed); out = []
    while len(out) < n:
        a, b = rng.choice(NAMES, size=2, replace=False)
        t = TEMPLATES[rng.integers(len(TEMPLATES))]
        out.append((t.format(A=a,B=b), t.format(A=b,B=a), " "+a, " "+b))
    return out

def logit_diff_final(logits, io_toks, s_toks, lengths):
    idx = torch.arange(logits.shape[0], device=logits.device)
    final = logits[idx, lengths - 1]
    return (final[idx, io_toks] - final[idx, s_toks]).mean()

def prep_chunk(model, chunk):
    ct  = model.to_tokens([p[0] for p in chunk])
    cot = model.to_tokens([p[1] for p in chunk])
    cl_len = torch.tensor([model.to_tokens(p[0]).shape[1] for p in chunk], device=DEVICE)
    co_len = torch.tensor([model.to_tokens(p[1]).shape[1] for p in chunk], device=DEVICE)
    io = torch.tensor([model.to_single_token(p[2]) for p in chunk], device=DEVICE)
    s  = torch.tensor([model.to_single_token(p[3]) for p in chunk], device=DEVICE)
    return ct, cot, cl_len, co_len, io, s

def tracked_patching(model, seed, heads):
    pairs = make_pairs(N_PAIRS, seed)
    chunks = [pairs[i:i+25] for i in range(0, N_PAIRS, 25)]
    chunk_data, clean_lds, corr_lds = [], [], []
    for chunk in chunks:
        ct, cot, cl_len, co_len, io, s = prep_chunk(model, chunk)
        with torch.no_grad():
            cl_logits, cl_cache = model.run_with_cache(ct)
            co_logits = model(cot)
        chunk_data.append((cot, co_len, io, s, cl_cache))
        clean_lds.append(logit_diff_final(cl_logits, io, s, cl_len).item())
        corr_lds.append(logit_diff_final(co_logits, io, s, co_len).item())
    clean_ld, corr_ld = np.mean(clean_lds), np.mean(corr_lds)

    def patch_head(z, hook, head, layer, cache):
        z[:, :, head, :] = cache[f"blocks.{layer}.attn.hook_z"][:, :, head, :]
        return z

    out = {}
    for (layer, head) in heads:
        lds = []
        for (cot, co_len, io, s, cl_cache) in chunk_data:
            hook_fn = partial(patch_head, head=head, layer=layer, cache=cl_cache)
            with torch.no_grad():
                patched = model.run_with_hooks(
                    cot, fwd_hooks=[(f"blocks.{layer}.attn.hook_z", hook_fn)])
            lds.append(logit_diff_final(patched, io, s, co_len).item())
        out[(layer, head)] = np.mean(lds) - corr_ld
    return out, clean_ld, corr_ld

# ---------- SANITY GATE: FP32 seed 0 must match known-good ----------
print("===== SANITY GATE (FP32, seed 0) =====")
model = make_model(None)
gate, cl, co = tracked_patching(model, 0, ALL_TRACKED)
del model; torch.cuda.empty_cache()
print(f"clean {cl:+.3f} (expect ~+4) | corrupted {co:+.3f} (expect ~-4)")
print(f"10.7: {gate[(10,7)]:+.3f} (expect ~-0.8) | 11.10: {gate[(11,10)]:+.3f} (expect strongly negative)")
spread = cl - co
ok = (cl > 2.5) and (-0.60 < gate[(10,7)]/spread < -0.30) and (gate[(11,10)]/spread < -0.15)
print("GATE:", "PASS — continuing" if ok else "FAIL — STOPPING, do not trust results")
assert ok, "Baseline does not match exp2 known-good values."

# ---------- full run ----------
levels = [("FP32", None), ("INT6", 6), ("INT5", 5), ("INT4", 4)]
raw = {}
for label, bits in levels:
    for seed in SEEDS:
        print(f"=== {label} seed {seed} ===")
        model = make_model(bits)
        raw[(label, seed)], _, _ = tracked_patching(model, seed, ALL_TRACKED)
        del model; torch.cuda.empty_cache()

print("\n===== FP32 RAW EFFECTS (mean±std) =====")
for (l,h) in ALL_TRACKED:
    vals = [raw[("FP32",s)][(l,h)] for s in SEEDS]
    print(f"  {l}.{h}: {np.mean(vals):+.3f} ± {np.std(vals):.3f}")

print("\n===== RETENTION =====")
print(f"{'head':8s} {'type':4s} " + " ".join(f"{lb:>6s}" for lb, _ in levels[1:]))
retention = {lb: {"pos": [], "neg": []} for lb, _ in levels[1:]}
for (l, h) in ALL_TRACKED:
    kind = "pos" if (l, h) in POSITIVE else "neg"
    fp_mean = np.mean([abs(raw[("FP32", s)][(l,h)]) for s in SEEDS])
    row = f"{l}.{h:<6d} {kind:4s} "
    for lb, _ in levels[1:]:
        q_mean = np.mean([abs(raw[(lb, s)][(l,h)]) for s in SEEDS])
        r = q_mean / (fp_mean + 1e-9)
        retention[lb][kind].append(r); row += f" {r:6.2f}"
    print(row)

print("\n===== GROUP MEANS + TEST =====")
for lb, _ in levels[1:]:
    pos, neg = retention[lb]["pos"], retention[lb]["neg"]
    u, p = stats.mannwhitneyu(pos, neg, alternative="greater")
    print(f"{lb}: pos retain {np.mean(pos):.2f} | neg retain {np.mean(neg):.2f} | p={p:.4f}")

===== SANITY GATE (FP32, seed 0) =====
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
clean +2.984 (expect ~+4) | corrupted -2.998 (expect ~-4)
10.7: -2.562 (expect ~-0.8) | 11.10: -1.703 (expect strongly negative)
GATE: PASS — continuing
=== FP32 seed 0 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
=== FP32 seed 1 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
=== FP32 seed 2 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
=== INT6 seed 0 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
=== INT6 seed 1 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
=== INT6 seed 2 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
=== INT5 seed 0 ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
=== INT5 see

In [7]:
"""
Exp 4 — Is circuit damage non-monotonic in quantization strength?
Fine sweep: 16 grid densities from INT6 down to INT4.
Tracks: weight error (must be monotonic) vs functional effect of the
4 biggest heads (if non-monotonic, weight metrics can't predict damage).
Needs: make_pairs, logit_diff_final, prep_chunk, tracked_patching from exp3 v2.
"""
import torch, numpy as np
from functools import partial

DEVICE = "cuda"
HEADS = [(9,9), (10,0), (10,7), (11,10)]   # 2 biggest movers, 2 biggest suppressors
N_LEVELS_SWEEP = np.array([4.0, 4.13, 4.27])

def quantize_groupwise_levels(w, n_levels, group=128):
    """Same group-wise RTN but with arbitrary number of grid levels."""
    half = (n_levels - 1) / 2
    orig_shape = w.shape
    d_in = w.shape[-2]
    if d_in % group != 0:
        scale = w.abs().amax(dim=-2, keepdim=True) / half
        scale = scale.clamp(min=1e-12)
        return ((w/scale).round().clamp(-np.floor(half), np.floor(half)))*scale
    n_groups = d_in // group
    w2 = w.reshape(*w.shape[:-2], n_groups, group, w.shape[-1])
    scale = w2.abs().amax(dim=-2, keepdim=True) / half
    scale = scale.clamp(min=1e-12)
    q = (w2/scale).round().clamp(-np.floor(half), np.floor(half)) * scale
    return q.reshape(orig_shape)

QUANT_TARGETS = ("W_Q","W_K","W_V","W_O","W_in","W_out")

def make_model_levels(eff_bits=None):
    from transformer_lens import HookedTransformer
    model = HookedTransformer.from_pretrained("gpt2").to(DEVICE)
    if eff_bits is None: return model
    n_levels = 2 ** eff_bits
    with torch.no_grad():
        for name, p in model.named_parameters():
            if any(t in name for t in QUANT_TARGETS):
                p.copy_(quantize_groupwise_levels(p.data, n_levels))
    return model

# --- FP32 reference: weights + head effects ---
print("=== FP32 reference ===")
fp_model = make_model_levels(None)
w_ref = {n: p.data.clone() for n, p in fp_model.named_parameters()
         if any(t in n for t in QUANT_TARGETS)}
fp_eff, _, _ = tracked_patching(fp_model, 0, HEADS)
del fp_model; torch.cuda.empty_cache()

# --- sweep (seed 0 only for speed; we replicate winners later) ---
rows = []
for eb in N_LEVELS_SWEEP:
    model = make_model_levels(eb)
    errs = [((p.data - w_ref[n]).norm() / w_ref[n].norm()).item()
            for n, p in model.named_parameters() if n in w_ref]
    for seed in [0, 1, 2]:
        eff, cl, co = tracked_patching(model, seed, HEADS)
        row = {"eff_bits": round(float(eb),2), "seed": seed,
               "weight_err": round(float(np.mean(errs)),4), "clean_ld": round(cl,3)}
        for (l,h) in HEADS:
            row[f"{l}.{h}"] = round(eff[(l,h)] / (fp_eff[(l,h)] + 1e-9), 2)
        rows.append(row); print(row)
    del model; torch.cuda.empty_cache()

import pandas as pd
df = pd.DataFrame(rows)
df.to_csv("results/exp4_sweep.csv", index=False)
print("\n", df.to_string(index=False))
print("""
READ:
- weight_err column MUST decrease smoothly as eff_bits rises (quantizer sanity).
- Head columns: 1.0 = intact. If a head's retention dips at ~5.0 and RECOVERS
  at 4.5 -> non-monotonic damage confirmed on a fine grid, not a 2-point fluke.
- clean_ld shows whether task performance itself wobbles non-monotonically.
""")

=== FP32 reference ===
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
{'eff_bits': 4.0, 'seed': 0, 'weight_err': 0.126, 'clean_ld': np.float64(1.881), '9.9': np.float64(0.55), '10.0': np.float64(0.65), '10.7': np.float64(0.63), '11.10': np.float64(0.15)}
{'eff_bits': 4.0, 'seed': 1, 'weight_err': 0.126, 'clean_ld': np.float64(1.931), '9.9': np.float64(0.56), '10.0': np.float64(0.58), '10.7': np.float64(0.6), '11.10': np.float64(-0.01)}
{'eff_bits': 4.0, 'seed': 2, 'weight_err': 0.126, 'clean_ld': np.float64(2.204), '9.9': np.float64(0.65), '10.0': np.float64(0.68), '10.7': np.float64(0.76), '11.10': np.float64(0.08)}
Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
{'eff_bits': 4.13, 'seed': 0, 'weight_err': 0.1136, 'clean_ld': np.float64(1.187), '9.9': np.float64(0.32), '10.0': np.float64(0.25), '10.7': np.float64(0.02), '11.10': np.floa

In [8]:
"""
Exp 5 — Pythia-70M: does spiky, weight-error-defying damage generalize?
Pipeline: (1) FP32 full patching map -> find ITS top heads
          (2) fine bit sweep 4.0-6.0, tracking those heads
          (3) flag any grid where damage defies weight error
Needs: make_pairs, logit_diff_final in scope. Self-contained otherwise.
"""
import torch, numpy as np, pandas as pd
from functools import partial
from transformer_lens import HookedTransformer

DEVICE = "cuda"
MODEL_NAME = "pythia-160m"
N_PAIRS = 100
QUANT_TARGETS = ("W_Q","W_K","W_V","W_O","W_in","W_out")

def quantize_groupwise_levels(w, n_levels, group=64):   # d_model=512 -> group 64
    half = (n_levels - 1) / 2
    orig_shape = w.shape
    d_in = w.shape[-2]
    if d_in % group != 0:
        scale = w.abs().amax(dim=-2, keepdim=True) / half
        scale = scale.clamp(min=1e-12)
        return ((w/scale).round().clamp(-np.floor(half), np.floor(half)))*scale
    n_groups = d_in // group
    w2 = w.reshape(*w.shape[:-2], n_groups, group, w.shape[-1])
    scale = w2.abs().amax(dim=-2, keepdim=True) / half
    scale = scale.clamp(min=1e-12)
    q = (w2/scale).round().clamp(-np.floor(half), np.floor(half)) * scale
    return q.reshape(orig_shape)

def make_pythia(eff_bits=None):
    model = HookedTransformer.from_pretrained(MODEL_NAME).to(DEVICE)
    if eff_bits is None: return model
    n_levels = 2 ** eff_bits
    with torch.no_grad():
        for name, p in model.named_parameters():
            if any(t in name for t in QUANT_TARGETS):
                p.copy_(quantize_groupwise_levels(p.data, n_levels))
    return model

def prep_chunk(model, chunk):
    ct  = model.to_tokens([p[0] for p in chunk])
    cot = model.to_tokens([p[1] for p in chunk])
    cl_len = torch.tensor([model.to_tokens(p[0]).shape[1] for p in chunk], device=DEVICE)
    co_len = torch.tensor([model.to_tokens(p[1]).shape[1] for p in chunk], device=DEVICE)
    io = torch.tensor([model.to_single_token(p[2]) for p in chunk], device=DEVICE)
    s  = torch.tensor([model.to_single_token(p[3]) for p in chunk], device=DEVICE)
    return ct, cot, cl_len, co_len, io, s

def full_map(model, seed):
    """Patch every head. Returns raw restoration map + clean/corr lds."""
    pairs = make_pairs(N_PAIRS, seed)
    chunks = [pairs[i:i+25] for i in range(0, N_PAIRS, 25)]
    data, cls, cos = [], [], []
    for chunk in chunks:
        ct, cot, cl_len, co_len, io, s = prep_chunk(model, chunk)
        with torch.no_grad():
            cl_logits, cl_cache = model.run_with_cache(ct)
            co_logits = model(cot)
        data.append((cot, co_len, io, s, cl_cache))
        cls.append(logit_diff_final(cl_logits, io, s, cl_len).item())
        cos.append(logit_diff_final(co_logits, io, s, co_len).item())
    clean_ld, corr_ld = np.mean(cls), np.mean(cos)

    def patch_head(z, hook, head, layer, cache):
        z[:, :, head, :] = cache[f"blocks.{layer}.attn.hook_z"][:, :, head, :]
        return z

    n_l, n_h = model.cfg.n_layers, model.cfg.n_heads
    m = np.zeros((n_l, n_h))
    for layer in range(n_l):
        for head in range(n_h):
            lds = []
            for (cot, co_len, io, s, cl_cache) in data:
                hook_fn = partial(patch_head, head=head, layer=layer, cache=cl_cache)
                with torch.no_grad():
                    patched = model.run_with_hooks(
                        cot, fwd_hooks=[(f"blocks.{layer}.attn.hook_z", hook_fn)])
                lds.append(logit_diff_final(patched, io, s, co_len).item())
            m[layer, head] = np.mean(lds) - corr_ld
    return m, clean_ld, corr_ld

def tracked(model, seed, heads):
    pairs = make_pairs(N_PAIRS, seed)
    chunks = [pairs[i:i+25] for i in range(0, N_PAIRS, 25)]
    data, cls, cos = [], [], []
    for chunk in chunks:
        ct, cot, cl_len, co_len, io, s = prep_chunk(model, chunk)
        with torch.no_grad():
            cl_logits, cl_cache = model.run_with_cache(ct)
            co_logits = model(cot)
        data.append((cot, co_len, io, s, cl_cache))
        cls.append(logit_diff_final(cl_logits, io, s, cl_len).item())
        cos.append(logit_diff_final(co_logits, io, s, co_len).item())
    clean_ld, corr_ld = np.mean(cls), np.mean(cos)
    def patch_head(z, hook, head, layer, cache):
        z[:, :, head, :] = cache[f"blocks.{layer}.attn.hook_z"][:, :, head, :]
        return z
    out = {}
    for (layer, head) in heads:
        lds = []
        for (cot, co_len, io, s, cl_cache) in data:
            hook_fn = partial(patch_head, head=head, layer=layer, cache=cl_cache)
            with torch.no_grad():
                patched = model.run_with_hooks(
                    cot, fwd_hooks=[(f"blocks.{layer}.attn.hook_z", hook_fn)])
            lds.append(logit_diff_final(patched, io, s, co_len).item())
        out[(layer, head)] = np.mean(lds) - corr_ld
    return out, clean_ld, corr_ld

# ---------- STEP 1: does Pythia-70M even do IOI? ----------
print("===== STEP 1: FP32 baseline + circuit discovery =====")
fp = make_pythia(None)
m, cl, co = full_map(fp, 0)
print(f"clean {cl:+.3f} | corrupted {co:+.3f}")
if cl < 0.5:
    print("WARNING: Pythia-160M may be too weak at IOI. If clean_ld ~ 0, STOP -> we switch to pythia-160m.")
flat = [(-abs(m[l,h]), l, h, m[l,h]) for l in range(m.shape[0]) for h in range(m.shape[1])]
top8 = [(l, h, round(v,3)) for _, l, h, v in sorted(flat)[:8]]
print("top heads:", top8)
HEADS = [(l,h) for l,h,_ in top8[:6]]
w_ref = {n: p.data.clone() for n, p in fp.named_parameters()
         if any(t in n for t in QUANT_TARGETS)}
fp_eff, _, _ = tracked(fp, 0, HEADS)
del fp; torch.cuda.empty_cache()
assert cl > 1.0, f"clean_ld={cl:.2f} — model can't do IOI, escalate to pythia-410m"

# ---------- STEP 2: fine sweep ----------
print("\n===== STEP 2: bit sweep =====")
rows = []
for eb in np.linspace(4.0, 6.0, 16):
    model = make_pythia(eb)
    errs = [((p.data - w_ref[n]).norm() / w_ref[n].norm()).item()
            for n, p in model.named_parameters() if n in w_ref]
    eff, c, _ = tracked(model, 0, HEADS)
    row = {"eff_bits": round(float(eb),2), "weight_err": round(float(np.mean(errs)),4),
           "clean_ld": round(c,3)}
    for (l,h) in HEADS:
        row[f"{l}.{h}"] = round(eff[(l,h)] / (fp_eff[(l,h)] + 1e-9), 2)
    rows.append(row); print(row)
    del model; torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv("results/exp5_pythia_sweep.csv", index=False)
print("\n", df.to_string(index=False))
print("""
READ:
1. If STEP 1 clean_ld ~ 0: stop, Pythia-70M can't do IOI, we move to 160m.
2. weight_err must fall smoothly.
3. Look for any bit level where retention crashes vs both neighbors
   -> spiky damage generalizes. The level will likely DIFFER from GPT-2's
   4.13 (grids interact with different weights) — that's expected and fine.
""")

===== STEP 1: FP32 baseline + circuit discovery =====


config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/375M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
clean +2.608 | corrupted -2.188
top heads: [(8, 9, np.float64(2.718)), (3, 2, np.float64(2.469)), (10, 7, np.float64(0.687)), (3, 0, np.float64(0.653)), (8, 10, np.float64(0.496)), (6, 5, np.float64(0.419)), (9, 1, np.float64(-0.364)), (9, 7, np.float64(0.327))]

===== STEP 2: bit sweep =====
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'eff_bits': 4.0, 'weight_err': 0.1101, 'clean_ld': np.float64(1.557), '8.9': np.float64(1.02), '3.2': np.float64(0.85), '10.7': np.float64(0.32), '3.0': np.float64(1.14), '8.10': np.float64(-0.07), '6.5': np.float64(1.0)}
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'eff_bits': 4.13, 'weight_err': 0.0982, 'clean_ld': np.float64(1.961), '8.9': np.float64(0.58), '3.2': np.float64(0.75), '10.7': np.float64(1.24), '3.0': np.float64(0.81), '8.10': np.float64(0.37), '6.5': np.float64

In [9]:
"""
Exp 5b — Confirm Pythia-160m's 8.9 crater across 3 seeds.
Grids: 4.0 (healthy), 4.13, 4.27, 4.40 (crater), 4.53 (recovery edge).
Needs: make_pairs, logit_diff_final, and exp5's make_pythia, tracked,
quantize_groupwise_levels in scope. MODEL_NAME must be "pythia-160m".
"""
import torch, numpy as np, pandas as pd

HEADS = [(8,9), (3,2), (10,7), (3,0), (6,5)]   # drop 8.10: too small/unstable to ratio
GRIDS = [4.0, 4.13, 4.27, 4.40, 4.53]
SEEDS = [0, 1, 2]

# FP32 reference per seed (ratios computed seed-to-seed, cleaner than one ref)
print("=== FP32 reference (3 seeds) ===")
fp = make_pythia(None)
fp_eff = {}
for s in SEEDS:
    fp_eff[s], cl, _ = tracked(fp, s, HEADS)
    print(f"seed {s}: clean {cl:+.3f} | " +
          " ".join(f"{l}.{h}:{fp_eff[s][(l,h)]:+.2f}" for l,h in HEADS))
del fp; torch.cuda.empty_cache()

rows = []
for eb in GRIDS:
    model = make_pythia(eb)
    for s in SEEDS:
        eff, cl, _ = tracked(model, s, HEADS)
        row = {"eff_bits": eb, "seed": s, "clean_ld": round(cl,3)}
        for (l,h) in HEADS:
            row[f"{l}.{h}"] = round(eff[(l,h)] / (fp_eff[s][(l,h)] + 1e-9), 2)
        rows.append(row); print(row)
    del model; torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv("results/exp5b_pythia_confirm.csv", index=False)
print("\n", df.to_string(index=False))
print("\n===== 8.9 MEAN RETENTION PER GRID =====")
for eb in GRIDS:
    vals = df[df.eff_bits==eb]["8.9"].values
    print(f"{eb}: {np.mean(vals):.2f} ± {np.std(vals):.2f}")

=== FP32 reference (3 seeds) ===
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
seed 0: clean +2.608 | 8.9:+2.72 3.2:+2.47 10.7:+0.69 3.0:+0.65 6.5:+0.42
seed 1: clean +1.960 | 8.9:+2.40 3.2:+2.50 10.7:+0.58 3.0:+0.53 6.5:+0.46
seed 2: clean +2.414 | 8.9:+2.76 3.2:+2.95 10.7:+0.64 3.0:+0.60 6.5:+0.46
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'eff_bits': 4.0, 'seed': 0, 'clean_ld': np.float64(1.557), '8.9': np.float64(1.02), '3.2': np.float64(0.85), '10.7': np.float64(0.32), '3.0': np.float64(1.14), '6.5': np.float64(1.0)}
{'eff_bits': 4.0, 'seed': 1, 'clean_ld': np.float64(1.308), '8.9': np.float64(1.12), '3.2': np.float64(0.78), '10.7': np.float64(0.24), '3.0': np.float64(1.26), '6.5': np.float64(0.85)}
{'eff_bits': 4.0, 'seed': 2, 'clean_ld': np.float64(1.771), '8.9': np.float64(1.12), '3.2': np.float64(0.76), '10.7': np.float64(0.42), '3.0': np.float64(1.46), '6.5': np.float64(1.09)}
Loaded pre

In [10]:
"""
Exp 6 — Localize the wound. Pythia-160m, head 8.9, killer grid 4.27.
Quantize the WHOLE model, then restore ONE weight matrix of head 8.9
to full precision at a time. Which restoration revives the head?
Also: restore all 4 at once (is the wound inside 8.9 at all?).
Needs: make_pythia, tracked, make_pairs, logit_diff_final in scope.
"""
import torch, numpy as np, pandas as pd
from transformer_lens import HookedTransformer

DEVICE = "cuda"
LAYER, HEAD = 8, 9
KILLER = 4.27
SEEDS = [0, 1, 2]
HEADS = [(8,9)]

# FP32 reference: head effect + weight slices for layer 8
print("=== FP32 reference ===")
fp = make_pythia(None)
fp_eff = {}
for s in SEEDS:
    fp_eff[s], _, _ = tracked(fp, s, HEADS)
fp_weights = {}
for name, p in fp.named_parameters():
    if f"blocks.{LAYER}.attn" in name and any(t in name for t in ("W_Q","W_K","W_V","W_O")):
        fp_weights[name] = p.data.clone()
print("saved FP32 slices:", list(fp_weights.keys()))
del fp; torch.cuda.empty_cache()

def restore(model, which):
    """Restore head HEAD's slice of the given matrices to FP32.
    Pythia TL shapes: W_Q/K/V [n_heads, d_model, d_head], W_O [n_heads, d_head, d_model]."""
    with torch.no_grad():
        for name, p in model.named_parameters():
            for w in which:
                if name in fp_weights and name.endswith(w):
                    p.data[HEAD] = fp_weights[name][HEAD]
    return model

conditions = [
    ("none (full quant)", []),
    ("restore W_Q", ["W_Q"]),
    ("restore W_K", ["W_K"]),
    ("restore W_V", ["W_V"]),
    ("restore W_O", ["W_O"]),
    ("restore ALL 4", ["W_Q","W_K","W_V","W_O"]),
]

rows = []
for label, which in conditions:
    model = make_pythia(KILLER)
    model = restore(model, which)
    rets = []
    for s in SEEDS:
        eff, cl, _ = tracked(model, s, HEADS)
        rets.append(eff[(LAYER,HEAD)] / (fp_eff[s][(LAYER,HEAD)] + 1e-9))
    rows.append({"condition": label,
                 "retention_mean": round(float(np.mean(rets)),2),
                 "retention_std": round(float(np.std(rets)),2)})
    print(rows[-1])
    del model; torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv("results/exp6_localize.csv", index=False)
print("\n", df.to_string(index=False))
print("""
READ:
- baseline 'none' should be ~0.20 (the crater).
- If ONE matrix restoration jumps retention to ~1.0 -> wound localized there.
- If only 'ALL 4' revives it -> distributed damage inside the head.
- If even 'ALL 4' stays low -> the wound is UPSTREAM (8.9 is healthy but
  receiving corrupted inputs from earlier layers). Also a clean finding.
""")

=== FP32 reference ===
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
saved FP32 slices: ['blocks.8.attn.W_Q', 'blocks.8.attn.W_O', 'blocks.8.attn.W_K', 'blocks.8.attn.W_V']
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'condition': 'none (full quant)', 'retention_mean': 0.2, 'retention_std': 0.02}
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'condition': 'restore W_Q', 'retention_mean': 0.41, 'retention_std': 0.04}
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'condition': 'restore W_K', 'retention_mean': 0.76, 'retention_std': 0.04}
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'condition': 'restore W_V', 'retention_mean': 0.2, 'retention_std': 0.02}
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'condition': 'restore W_O', 'r

In [11]:
"""
Exp 8 — WHY: where does the killer grid's damage live?
For each grid, measure on head 8.9:
  (1) raw weight error of W_Q, W_K            (known: smooth)
  (2) error of the FUNCTIONAL object Q·K^T    (prediction: spiky, tracks retention)
  (3) how much of W_K's error lies in its top-5 important directions (SVD)
Needs: make_pythia in scope.
"""
import torch, numpy as np, pandas as pd

LAYER, HEAD = 8, 9
GRIDS = [4.0, 4.13, 4.27, 4.4, 4.53, 4.67, 4.8]
RETENTION = {4.0:1.09, 4.13:0.86, 4.27:0.20, 4.4:0.26, 4.53:0.60}  # from exp5b

# FP32 reference
fp = make_pythia(None)
Wq_fp = Wk_fp = None
for n, p in fp.named_parameters():
    if f"blocks.{LAYER}.attn.W_Q" in n: Wq_fp = p.data[HEAD].clone()  # [d_model, d_head]
    if f"blocks.{LAYER}.attn.W_K" in n: Wk_fp = p.data[HEAD].clone()
QK_fp = Wq_fp @ Wk_fp.T                      # [d_model, d_model] functional object
U, S, Vt = torch.linalg.svd(Wk_fp, full_matrices=False)
U5 = U[:, :5]                                # top-5 important directions of W_K
del fp; torch.cuda.empty_cache()

rows = []
for g in GRIDS:
    m = make_pythia(g)
    for n, p in m.named_parameters():
        if f"blocks.{LAYER}.attn.W_Q" in n: Wq = p.data[HEAD]
        if f"blocks.{LAYER}.attn.W_K" in n: Wk = p.data[HEAD]
    Ek, Eq = Wk - Wk_fp, Wq - Wq_fp
    QK_err = ((Wq @ Wk.T) - QK_fp).norm() / QK_fp.norm()
    align = (U5.T @ Ek).norm()**2 / Ek.norm()**2   # fraction of K-error in top-5 dirs
    rows.append({"grid": g,
                 "K_weight_err": round((Ek.norm()/Wk_fp.norm()).item(), 4),
                 "QK_product_err": round(QK_err.item(), 4),
                 "K_err_in_top5": round(align.item(), 4),
                 "retention": RETENTION.get(g, None)})
    print(rows[-1])
    del m; torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv("results/exp8_mechanism.csv", index=False)
print("\n", df.to_string(index=False))

Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'grid': 4.0, 'K_weight_err': 0.1051, 'QK_product_err': 0.5128, 'K_err_in_top5': 0.0083, 'retention': 1.09}
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'grid': 4.13, 'K_weight_err': 0.0938, 'QK_product_err': 0.4746, 'K_err_in_top5': 0.0072, 'retention': 0.86}
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'grid': 4.27, 'K_weight_err': 0.0845, 'QK_product_err': 0.43, 'K_err_in_top5': 0.0071, 'retention': 0.2}
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'grid': 4.4, 'K_weight_err': 0.0767, 'QK_product_err': 0.3946, 'K_err_in_top5': 0.0059, 'retention': 0.26}
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'grid': 4.53, 'K_weight_err': 0.06

In [12]:
"""
Exp 9 — Activation-level mechanism: does the error in head 8.9's ACTUAL
attention scores (post-rotary, real prompts) spike at the killer grid?
For each grid: run 50 IOI prompts, capture attention scores at the final
position, compare to FP32's scores.
Needs: make_pythia, make_pairs in scope.
"""
import torch, numpy as np, pandas as pd

DEVICE = "cuda"
LAYER, HEAD = 8, 9
GRIDS = [4.0, 4.13, 4.27, 4.4, 4.53]
RETENTION = {4.0:1.09, 4.13:0.86, 4.27:0.20, 4.4:0.26, 4.53:0.60}
N = 50

pairs = make_pairs(N, seed=0)
PROMPTS = [p[0] for p in pairs]

def get_scores(model):
    """Returns per-prompt: (final-row attention scores, final-row pattern,
    and John#2 position). Uses single prompts to avoid padding issues."""
    rows_scores, rows_pattern, dup_pos = [], [], []
    for text in PROMPTS:
        tokens = model.to_tokens(text)
        strs = model.to_str_tokens(text)
        _, cache = model.run_with_cache(tokens)
        sc = cache[f"blocks.{LAYER}.attn.hook_attn_scores"][0, HEAD, -1].cpu()  # [seq]
        pt = cache[f"blocks.{LAYER}.attn.hook_pattern"][0, HEAD, -1].cpu()
        # duplicate name = the token of the SECOND occurrence of the B name
        # B name is the one appearing twice; find any token string occurring twice
        seen, dpos = {}, None
        for i, s in enumerate(strs):
            if s.startswith(" ") and s[1:].isalpha() and s[1:][0].isupper():
                if s in seen: dpos = i
                seen[s] = i
        rows_scores.append(sc); rows_pattern.append(pt); dup_pos.append(dpos)
    return rows_scores, rows_pattern, dup_pos

print("=== FP32 reference ===")
fp = make_pythia(None)
fp_sc, fp_pt, dup = get_scores(fp)
del fp; torch.cuda.empty_cache()

rows = []
for g in GRIDS:
    m = make_pythia(g)
    sc, pt, _ = get_scores(m)
    # (a) mean L2 error of final-row scores vs FP32
    score_err = np.mean([ (sc[i]-fp_sc[i]).norm().item() / (fp_sc[i].norm().item()+1e-9)
                          for i in range(N) ])
    # (b) mean attention mass on the duplicate name (the head's target)
    dup_attn    = np.mean([ pt[i][dup[i]].item()    for i in range(N) if dup[i] is not None])
    dup_attn_fp = np.mean([ fp_pt[i][dup[i]].item() for i in range(N) if dup[i] is not None])
    rows.append({"grid": g,
                 "score_err": round(score_err, 3),
                 "dup_attention": round(dup_attn, 3),
                 "dup_attention_FP32": round(dup_attn_fp, 3),
                 "retention": RETENTION[g]})
    print(rows[-1])
    del m; torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv("results/exp9_activation_mechanism.csv", index=False)
print("\n", df.to_string(index=False))
r = np.corrcoef(df.dup_attention, df.retention)[0,1]
print(f"\ncorrelation(dup_attention, retention) = {r:.3f}")

=== FP32 reference ===
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'grid': 4.0, 'score_err': np.float64(0.604), 'dup_attention': np.float64(0.572), 'dup_attention_FP32': np.float64(0.706), 'retention': 1.09}
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'grid': 4.13, 'score_err': np.float64(0.677), 'dup_attention': np.float64(0.478), 'dup_attention_FP32': np.float64(0.706), 'retention': 0.86}
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'grid': 4.27, 'score_err': np.float64(1.313), 'dup_attention': np.float64(0.062), 'dup_attention_FP32': np.float64(0.706), 'retention': 0.2}
Loaded pretrained model pythia-160m into HookedTransformer
Moving model to device:  cuda
{'grid': 4.4, 'score_err': np.float64(0.677), 'dup_attention': np.float64(0.15), 'dup_attention_FP32': np